In [1]:
class Value:
    def __init__(self, data, prev=()):
        self.data = data
        self.grad = 0
        self.prev = prev
        self._backward = lambda:None
    
    def backward(self):
        topo = []
        visited = set()
        
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for n in v.prev:
                    build_topo(n)
                topo.append(v)
                
        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            v._backward()
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)        
        out = Value(self.data + other.data, (self, other))
        
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1    
    
    def __sub__(self, other):
        return self + (-other)
        
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)        
        out = Value(self.data * other.data, (self, other))
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        
        out._backward = _backward
        return out
    
    def __pow__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data ** other.data, (self,))       
        def _backward():
            self.grad += (other.data * self.data ** (other.data - 1)) * out.grad
        out._backward = _backward
        return out
    
    def relu(self):
        out = Value(self.data, (self,)) if self.data > 0 else Value(0, (self,))
        
        def _backward():
            self.grad += (self.data > 0) * out.grad
        
        out._backward = _backward        
        return out            

In [2]:
import random

class LinearLayer:
    def __init__(self, in_feat, out_feat):
        self.W = [[Value(random.uniform(-1, 1)) for _ in range(out_feat)] for _ in range(in_feat)]
        self.bias = [Value(0) for _ in range(out_feat)]
        self.shape = (in_feat, out_feat)
    
    def forward(self, X):
        out = [[Value(0) for _ in range(self.shape[1])] for _ in range(len(X))]
        
        for X_row_idx, row in enumerate(X):
            for W_col_idx in range(self.shape[1]):                
                for W_row_idx in range(self.shape[0]):            
                    out[X_row_idx][W_col_idx] += self.W[W_row_idx][W_col_idx] * row[W_row_idx] 
            
        for row in out:
            for i in range(self.shape[1]):
                row[i] += self.bias[i]              
                              
        return out
    
    def zero_grad(self):
        for row in self.W:
            for p in row:
                p.grad = 0
        for p in self.bias:
            p.grad = 0

In [3]:
lin = LinearLayer(2, 5)
lin2 = LinearLayer(5, 1)

In [4]:
out = lin.forward([[Value(4), Value(3)]])

In [5]:
mults = [1,2,3,4,5,6,7,8,9]
for i in range(10000):
    lin.zero_grad()
    lin2.zero_grad()
    x = Value(random.choice(mults))
    y = Value(random.choice(mults))
    out = lin.forward([[x, y]])
    out = [[v.relu() for v in row] for row in out]
    out = lin2.forward(out)
    y_pred = out[0][0]        
    y_true = x * y    
    loss = (y_true - y_pred)**2
    if (i % 100 == 0):    
        print(loss.data) 
    loss.backward()
    
    lr = 1e-3
    for row in lin.W:
        for p in row:
            p.data += (-p.grad) * lr
    for p in lin.bias:
        p.data += (-p.grad) * lr      

877.4841549959331
126.50363473419365
385.36767601821373
4.110979199729019
105.16401091640391
5.823899191071894
135.71915894887348
11.05495057503134
16.66519199818989
5.4192663987548615
90.5599634879324
59.77989609496074
22.477578031816012
21.405314481063236
43.14047378457913
150.79351096838747
4.577267843500223
4.287260126682055
172.2591919057769
87.8152568470441
239.2363677901485
7.375634406870949
7.883108239804177
205.27430780470596
8.823299204396582
2.7757507147188267
27.59053938120828
3.582792406272115
105.2514694184807
0.5359317867926267
27.256726962727665
5.484414447589095
45.484884056172014
1.3936366547834689
0.0026057784457787594
0.4022908581182154
33.541849865144464
27.483349738634224
20.85622084275107
0.5556496849938222
0.33261608565064804
2.8953804342360874
0.023177118156490373
0.5088734266899724
6.948914722698696
0.13989140973580633
175.57481098129227
2.5653205215178154
0.8333432481887011
0.021269285124149023
1.509894852420259
199.75165194893714
43.422518501994695
0.0326560